# 🚀 MinerU PDF to JSON Converter
## Advanced Document Processing for Google Colab

This notebook demonstrates how to convert PDF files to structured JSON using **MinerU**, a powerful document parsing tool that preserves:

- ✅ **Text content** with proper reading order
- ✅ **Tables** in HTML format
- ✅ **Mathematical formulas** in LaTeX
- ✅ **Images and figures**
- ✅ **Document structure** and layout
- ✅ **Multi-language support** with OCR

---

### 📋 Table of Contents
1. [Setup & Installation](#setup)
2. [Upload Converter Script](#upload)
3. [Quick Start - Basic Conversion](#quickstart)
4. [Advanced Usage](#advanced)
5. [Analyzing Output](#analysis)
6. [Batch Processing](#batch)
7. [Troubleshooting](#troubleshooting)

---

**Author:** RAG Application Pipeline  
**Version:** 1.0  
**Last Updated:** October 2025

## 📦 Step 1: Setup and Installation

First, let's check the environment and install MinerU if needed.

In [ ]:
# Check Python version and environment
import sys
import os
from pathlib import Path

print(f"🐍 Python version: {sys.version}")
print(f"📂 Working directory: {os.getcwd()}")
print(f"💻 Running in: {'Google Colab' if 'google.colab' in sys.modules else 'Local Jupyter'}")

In [ ]:
# Install MinerU (this will take several minutes on first run)
print("📥 Installing MinerU and dependencies...")
print("⏳ This may download ~4GB of models on first run. Please be patient!\n")

!pip install -q -U mineru[core] --no-cache-dir

print("\n✅ Installation complete!")

## 📤 Step 2: Upload the Converter Script

Upload the `pdf_to_json_mineru_colab.py` file to your Colab environment.

In [ ]:
# Upload the converter script from your local machine
from google.colab import files
import shutil

print("📤 Please upload the 'pdf_to_json_mineru_colab.py' file...")
uploaded = files.upload()

# Move to working directory
if 'pdf_to_json_mineru_colab.py' in uploaded:
    print("✅ Script uploaded successfully!")
    
    # Make it executable
    os.chmod('pdf_to_json_mineru_colab.py', 0o755)
    print("✅ Script is ready to use!")
else:
    print("❌ Script not found. Please upload 'pdf_to_json_mineru_colab.py'")

In [ ]:
# Import the converter functions
from pdf_to_json_mineru_colab import convert_pdf, convert_pdf_to_json_mineru

print("✅ Converter functions imported successfully!")
print("\n📚 Available functions:")
print("  - convert_pdf(pdf_path, output_path, verbose, ocr_only)")
print("  - convert_pdf_to_json_mineru(pdf_path, output_path, config)")

## 📄 Step 3: Upload Your PDF File

Upload the PDF file you want to convert.

In [ ]:
# Upload your PDF file
print("📤 Please upload your PDF file...")
uploaded_pdfs = files.upload()

# Get the uploaded PDF filename
pdf_filename = list(uploaded_pdfs.keys())[0] if uploaded_pdfs else None

if pdf_filename:
    pdf_path = f"/content/{pdf_filename}"
    file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f"\n✅ PDF uploaded: {pdf_filename}")
    print(f"📊 File size: {file_size_mb:.2f} MB")
else:
    print("❌ No PDF file uploaded!")

## 🚀 Step 4: Quick Start - Basic Conversion

Let's convert your PDF to JSON with default settings.

In [ ]:
# Basic conversion - simple and straightforward
output_json = f"/content/{Path(pdf_filename).stem}_output.json"

print("🔄 Starting PDF conversion...")
print("⏳ This may take 5-15 minutes depending on PDF size and complexity\n")

# Run conversion
success = convert_pdf(pdf_path, output_json)

if success:
    print(f"\n🎉 Conversion completed!")
    print(f"📁 Output saved to: {output_json}")
else:
    print("\n❌ Conversion failed. Check the errors above.")

## 📊 Step 5: Analyze the Output

Let's examine what was extracted from your PDF.

In [ ]:
# Load and analyze the output JSON
import json

with open(output_json, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Display basic information
print("📋 DOCUMENT INFORMATION")
print("=" * 60)
print(f"Title: {data.get('title', 'N/A')}")
print(f"Source: {data.get('source', 'N/A')}")
print(f"Converter: {data.get('conversion_info', {}).get('converter', 'N/A')}")
print()

# Display statistics
stats = data.get('metadata', {}).get('statistics', {})
if stats:
    print("📊 EXTRACTION STATISTICS")
    print("=" * 60)
    print(f"Total text length: {stats.get('total_text_length', 0):,} characters")
    print(f"Content items: {stats.get('content_items', 0)}")
    print(f"Extracted images: {stats.get('extracted_images', 0)}")
    print(f"Tables found: {stats.get('tables_found', 0)}")
    print(f"Formulas found: {stats.get('formulas_found', 0)}")
    print()

# Show first 500 characters of extracted text
full_text = data.get('full_text', '')
if full_text:
    print("📝 TEXT PREVIEW (first 500 characters)")
    print("=" * 60)
    print(full_text[:500] + "..." if len(full_text) > 500 else full_text)
    print()

print(f"✅ Full data available in variable 'data'")

### 🔍 Extracting Tables

If your PDF contains tables, let's view them.

In [ ]:
# Extract and display tables
from IPython.display import HTML, display

tables = data.get('tables', [])

if tables:
    print(f"📊 Found {len(tables)} table(s)")
    print()
    
    for i, table in enumerate(tables[:3], 1):  # Show first 3 tables
        print(f"Table {i}:")
        print("-" * 60)
        
        # Try to display as HTML if available
        if isinstance(table, dict):
            if 'html' in table:
                display(HTML(table['html']))
            else:
                print(json.dumps(table, indent=2)[:500])
        else:
            print(str(table)[:500])
        print()
else:
    print("ℹ️ No tables found in this document.")

### 🔢 Extracting Formulas

Mathematical formulas are extracted in LaTeX format.

In [ ]:
# Extract and display formulas
formulas = data.get('formulas', [])

if formulas:
    print(f"🔢 Found {len(formulas)} formula(s)")
    print()
    
    for i, formula in enumerate(formulas[:5], 1):  # Show first 5 formulas
        print(f"Formula {i}:")
        print("-" * 60)
        if isinstance(formula, dict):
            latex = formula.get('latex', formula.get('text', ''))
            print(f"LaTeX: {latex}")
        else:
            print(str(formula))
        print()
else:
    print("ℹ️ No mathematical formulas found in this document.")

### 🖼️ Viewing Extracted Images

Check what images were extracted from the PDF.

In [ ]:
# List extracted images
from IPython.display import Image, display
import matplotlib.pyplot as plt

images = data.get('extracted_images', [])

if images:
    print(f"🖼️ Found {len(images)} image(s)")
    print()
    
    for i, img_info in enumerate(images[:5], 1):  # Show first 5 images
        print(f"Image {i}:")
        print(f"  Filename: {img_info.get('filename', 'N/A')}")
        print(f"  Size: {img_info.get('size', 0):,} bytes")
        print(f"  Path: {img_info.get('path', 'N/A')}")
        
        # Try to display the image if path exists
        img_path = img_info.get('path')
        if img_path and os.path.exists(img_path):
            try:
                display(Image(filename=img_path, width=400))
            except:
                print("  (Could not display image)")
        print()
else:
    print("ℹ️ No images were extracted from this document.")

## ⚙️ Step 6: Advanced Usage

Let's explore advanced conversion options.

### 🔊 Verbose Mode

Get detailed logging during conversion to see what's happening.

In [ ]:
# Convert with verbose output to see detailed progress
output_verbose = "/content/output_verbose.json"

print("🔄 Converting with verbose output...\n")

success = convert_pdf(
    pdf_path=pdf_path,
    output_path=output_verbose,
    verbose=True  # Enable verbose logging
)

if success:
    print("\n✅ Verbose conversion completed!")
else:
    print("\n❌ Conversion failed.")

### 👁️ OCR-Only Mode (for Scanned PDFs)

If your PDF is a scanned document (images of pages), use OCR-only mode.

In [ ]:
# Convert scanned PDF using OCR-only mode
output_ocr = "/content/output_ocr.json"

print("🔄 Converting with OCR-only mode (for scanned PDFs)...\n")

success = convert_pdf(
    pdf_path=pdf_path,
    output_path=output_ocr,
    ocr_only=True  # Use OCR for scanned documents
)

if success:
    print("\n✅ OCR conversion completed!")
else:
    print("\n❌ Conversion failed.")

## 📦 Step 7: Batch Processing

Convert multiple PDFs at once.

In [ ]:
# Upload multiple PDFs
print("📤 Upload multiple PDF files for batch processing...")
batch_uploads = files.upload()

if batch_uploads:
    print(f"\n✅ Uploaded {len(batch_uploads)} file(s)")
    
    # Create output directory
    batch_output_dir = "/content/batch_outputs"
    os.makedirs(batch_output_dir, exist_ok=True)
    
    # Process each PDF
    results = []
    for i, pdf_file in enumerate(batch_uploads.keys(), 1):
        print(f"\n{'='*60}")
        print(f"Processing {i}/{len(batch_uploads)}: {pdf_file}")
        print(f"{'='*60}")
        
        input_path = f"/content/{pdf_file}"
        output_path = f"{batch_output_dir}/{Path(pdf_file).stem}_output.json"
        
        success = convert_pdf(input_path, output_path)
        results.append({
            'file': pdf_file,
            'success': success,
            'output': output_path if success else None
        })
    
    # Summary
    print(f"\n{'='*60}")
    print("📊 BATCH PROCESSING SUMMARY")
    print(f"{'='*60}")
    successful = sum(1 for r in results if r['success'])
    print(f"Total processed: {len(results)}")
    print(f"Successful: {successful}")
    print(f"Failed: {len(results) - successful}")
    print(f"\nOutputs saved to: {batch_output_dir}")
else:
    print("❌ No files uploaded!")

## 💾 Step 8: Download Your Results

Download the converted JSON files to your local machine.

In [ ]:
# Download the main output file
from google.colab import files

if os.path.exists(output_json):
    print(f"📥 Downloading: {output_json}")
    files.download(output_json)
    print("✅ Download initiated!")
else:
    print("❌ Output file not found. Please run the conversion first.")

In [ ]:
# Download all output files as a ZIP
import zipfile

zip_filename = "/content/mineru_outputs.zip"

# Find all JSON files in content directory
json_files = [f for f in os.listdir('/content') if f.endswith('_output.json')]

if json_files:
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for json_file in json_files:
            file_path = f"/content/{json_file}"
            zipf.write(file_path, json_file)
    
    print(f"📦 Created ZIP with {len(json_files)} file(s)")
    print(f"📥 Downloading: {zip_filename}")
    files.download(zip_filename)
    print("✅ Download complete!")
else:
    print("ℹ️ No output files found to zip.")

## 🔧 Troubleshooting & Tips

Common issues and solutions.

### Common Issues

**1. MinerU not found error**
- Solution: Re-run the installation  in the second cell

**2. Conversion takes too long**
- First run downloads ~4GB of AI models (layoutlmv3, unimernet, rapidtable, paddleocr)
- Subsequent conversions are much faster
- Large PDFs (>100 pages) can take 15-30 minutes

**3. OCR fails or produces poor results**
- Try using `ocr_only=True` for scanned documents
- Ensure PDF quality is good (at least 300 DPI for scans)
- Some handwritten text may not be recognized accurately

**4. Memory errors**
- Very large PDFs (>500 pages or >100MB) may cause issues
- Try processing smaller sections of the PDF
- Restart runtime and try again

**5. Missing tables or formulas**
- Check if the PDF actually contains these elements
- Some PDFs may have tables as images (use OCR mode)
- Formula detection works best with properly typeset mathematical content

### Performance Tips

- **Batch processing**: Process multiple similar PDFs together
- **Verbose mode**: Use `verbose=True` to see progress and identify bottlenecks
- **Output organization**: Create separate folders for different conversion batches
- **Validation**: Always check the statistics to ensure content was extracted

### Advanced Configuration

The script automatically sets up optimal MinerU configuration:
```json
{
  "layout": { "model": "layoutlmv3" },
  "formula": { "enable": true, "model": "unimernet" },
  "table": { "enable": true, "model": "rapidtable" },
  "ocr": { "enable": true, "model": "paddleocr" }
}
```

### Next Steps for RAG Applications

After conversion, you can:
1. **Chunk the text** for vector embeddings (typically 500-1000 characters)
2. **Create embeddings** using OpenAI, Cohere, or other embedding models
3. **Store in vector database** (Qdrant, Chroma, Pinecone, etc.)
4. **Build RAG pipeline** using LangChain or similar frameworks

### Resources

- **MinerU Documentation**: Check the official MinerU docs for advanced features
- **RAG Tutorial**: See the main project README for integration examples
- **LangChain Integration**: The `mineru_to_rag.py` script creates RAG-ready output

---

**Need Help?** Check the error messages carefully - they usually indicate what went wrong!

## 🎓 Bonus: Quick Reference

Copy-paste ready code snippets for common tasks.

In [ ]:
# QUICK REFERENCE - Copy these snippets as needed

# 1. Simple conversion
# convert_pdf('/content/your_file.pdf', '/content/output.json')

# 2. Verbose conversion (see detailed logs)
# convert_pdf('/content/your_file.pdf', '/content/output.json', verbose=True)

# 3. OCR-only mode (for scanned PDFs)
# convert_pdf('/content/scanned.pdf', '/content/output.json', ocr_only=True)

# 4. Load and inspect JSON output
# with open('/content/output.json', 'r') as f:
#     data = json.load(f)
#     print(f"Text length: {len(data['full_text'])}")
#     print(f"Tables: {len(data['tables'])}")
#     print(f"Images: {len(data['extracted_images'])}")

# 5. Extract just the text
# with open('/content/output.json', 'r') as f:
#     data = json.load(f)
#     full_text = data['full_text']
#     print(full_text)

# 6. Save text to separate file
# with open('/content/output.json', 'r') as f:
#     data = json.load(f)
# with open('/content/extracted_text.txt', 'w') as f:
#     f.write(data['full_text'])

# 7. Process directory of PDFs
# for pdf_file in os.listdir('/content/pdfs/'):
#     if pdf_file.endswith('.pdf'):
#         input_path = f'/content/pdfs/{pdf_file}'
#         output_path = f'/content/outputs/{pdf_file[:-4]}.json'
#         convert_pdf(input_path, output_path)

print("💡 Uncomment and modify the snippets above as needed!")

---

## 🎉 Congratulations!

You've successfully learned how to use MinerU to convert PDFs to structured JSON in Google Colab!

### What You've Learned:
✅ Install and configure MinerU  
✅ Convert PDFs with various options  
✅ Extract text, tables, formulas, and images  
✅ Handle batch processing  
✅ Download and use the results  

### Next Steps:
1. **For RAG Applications**: Use the extracted JSON to create vector embeddings
2. **For Data Analysis**: Process the structured data with pandas or other tools
3. **For Documentation**: Convert the JSON to other formats (Markdown, HTML, etc.)

---

**Happy Converting! 🚀**